# Pandas Practice 2
## Data Processing - ETL Pipeline
Reading a large CSV file from a URL, identifying stringified JSON columns, parsing and flattening nested data, cleaning and writing to a new CSV file.

In [2]:
import pandas as pd
import ast

In [3]:
url = "https://media.githubusercontent.com/media/BellexFire/Movies-ETL/refs/heads/main/movies_metadata.csv"
# Load the dataset from the provided URL
df = pd.read_csv(url, low_memory=False) #, low_memory=False to avoid dtype warning
df.shape

(45466, 24)

In [4]:
df.columns.tolist()

['adult',
 'belongs_to_collection',
 'budget',
 'genres',
 'homepage',
 'id',
 'imdb_id',
 'original_language',
 'original_title',
 'overview',
 'popularity',
 'poster_path',
 'production_companies',
 'production_countries',
 'release_date',
 'revenue',
 'runtime',
 'spoken_languages',
 'status',
 'tagline',
 'title',
 'video',
 'vote_average',
 'vote_count']

In [5]:
df.dtypes


adult                        str
belongs_to_collection        str
budget                       str
genres                       str
homepage                     str
id                           str
imdb_id                      str
original_language            str
original_title               str
overview                     str
popularity                   str
poster_path                  str
production_companies         str
production_countries         str
release_date                 str
revenue                  float64
runtime                  float64
spoken_languages             str
status                       str
tagline                      str
title                        str
video                     object
vote_average             float64
vote_count               float64
dtype: object

In [6]:
df_clean = df.copy()

In [9]:
df['popularity'] 

0        21.946943
1        17.015539
2          11.7129
3         3.859495
4         8.387519
           ...    
45461     0.072051
45462     0.178241
45463     0.903007
45464     0.003503
45465     0.163015
Name: popularity, Length: 45466, dtype: str

In [10]:
numreric_cols = ['budget', 'popularity', 'revenue', 'runtime', 'vote_average', 'vote_count']
for col in numreric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
df_clean['release_date'] = pd.to_datetime(df_clean['release_date'], errors='coerce')
df_clean.dtypes

adult                               str
belongs_to_collection               str
budget                          float64
genres                              str
homepage                            str
id                                  str
imdb_id                             str
original_language                   str
original_title                      str
overview                            str
popularity                      float64
poster_path                         str
production_companies                str
production_countries                str
release_date             datetime64[us]
revenue                         float64
runtime                         float64
spoken_languages                    str
status                              str
tagline                             str
title                               str
video                            object
vote_average                    float64
vote_count                      float64
dtype: object

### Parsing Stringified JSON Columns

In [31]:
df_clean['genres'] = df['genres'].copy()

df_clean['genres'] = df_clean['genres'].apply(
    lambda x: ' | '.join([genre['name'] for genre in ast.literal_eval(x)])
    if pd.notnull(x) and isinstance(x, str)  else ''
)



In [32]:
df_clean[['title', 'genres']].head()

,title,genres
0,Toy Story,Animation | Comedy | Family
1,Jumanji,Adventure | Fantasy | Family
2,Grumpier Old Men,Romance | Comedy
3,Waiting to Exhale,Comedy | Drama | Romance
4,Father of the Bride Part II,Comedy


In [35]:
df_clean['production_companies']= df_clean['production_companies'].apply(
    lambda x : '|'.join([company['name'] for company in ast.literal_eval(x)]) if pd.notnull(x) and isinstance(x, str) and x.strip().startswith('[') else ''
)
df_clean[['title', 'production_companies']].head()

,title,production_companies
0,Toy Story,Pixar Animation Studios
1,Jumanji,TriStar Pictures|Teitler Film|Interscope Commu...
2,Grumpier Old Men,Warner Bros.|Lancaster Gate
3,Waiting to Exhale,Twentieth Century Fox Film Corporation
4,Father of the Bride Part II,Sandollar Productions|Touchstone Pictures


In [38]:
df_clean['production_countries'] = df_clean['production_countries'].apply(
    lambda x : '|'.join([country['name'] for country in ast.literal_eval(x)]) if pd.notnull(x) and isinstance(x, str) and x.strip().startswith('[') else ''
)
df_clean[['title', 'production_countries']].head()

,title,production_countries
0,Toy Story,United States of America
1,Jumanji,United States of America
2,Grumpier Old Men,United States of America
3,Waiting to Exhale,United States of America
4,Father of the Bride Part II,United States of America


In [ ]:
df_clean['spoken_languages'] = df_clean['spoken_languages'].apply(
    lambda x : '|'.join([language['name'] for language in ast.literal_eval(x)]) if pd.notnull(x) and isinstance(x, str) and x.strip().startswith('[') else ''
)
df_clean.loc[:, ['title', 'spoken_languages']].head()

0                 [{'iso_639_1': 'en', 'name': 'English'}]
1        [{'iso_639_1': 'en', 'name': 'English'}, {'iso...
2                 [{'iso_639_1': 'en', 'name': 'English'}]
3                 [{'iso_639_1': 'en', 'name': 'English'}]
4                 [{'iso_639_1': 'en', 'name': 'English'}]
                               ...                        
45461               [{'iso_639_1': 'fa', 'name': 'فارسی'}]
45462                    [{'iso_639_1': 'tl', 'name': ''}]
45463             [{'iso_639_1': 'en', 'name': 'English'}]
45464                                                   []
45465             [{'iso_639_1': 'en', 'name': 'English'}]
Name: spoken_languages, Length: 45466, dtype: str